ST554_Final Project   
Author: Joy Zhou  
Date: 4/21/2026

# Introduction   
This project is designed to assess the ability to use Apache Spark for processing streaming data and for training machine learning models in a scalable computing environment.


We will use a modified dataset from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city) on power consumption in Tetouan City, the project focuses on modeling the relationship between electricity consumption across different city zones and key influencing factors such as time of day, temperature, and humidity.   


By leveraging Spark's data processing and machine learning libraries, a predictive model will be developed using historical data and then applied to incoming data streams to perform real-time monitoring and prediction of power consumption. This project integrates concepts from big data processing, machine learning, and streaming analytics, highlighting the practical application of Spark in handling real-world, data-intensive problems.


# Part 1 Fitting the Model



The dataset power_ml_data.csv is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv.
The data should first be loaded into a pandas DataFrame using the `pd.read_csv()` function and then converted into a Spark DataFrame. In this analysis, `Power_Zone_3` is treated as the response variable, while all remaining variables are used as predictors.

## Read in data

In [43]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

- First we read `power_ml_data` into a standard pandas data frame named `power_data` using the pd.read_csv() function.


In [44]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


- Convert `power_data` to a spark data frame `power`

In [45]:
#Convert this to a spark data frame
power = spark.createDataFrame(power_data)
power.show(5)
#We are going to treat the Power_Zone_3 variable as our response variable

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Data transformation
We will fit an elastic net regression using cross-validation (CV), without performing an explicit training-test split. Instead, model selection and performance assessment are conducted entirely through cross-validation on the available dataset.   

Feature transformations are performed using Spark MLlib utilities, and the transformed variables are assembled into a machine learning pipeline to ensure a consistent and reproducible workflow.

- let's check the varaible types by inspectiong the dataset schema.


In [47]:
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



### Hour variable binary transformation
- Inspection of the schema shows that the Hour column is defined as a LongType. We therefore apply an SQLTransformer to cast this variable to DoubleType.

In [48]:
from pyspark.ml.feature import SQLTransformer
sqlTrans = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

To inspect the results of the SQL transformation, we apply sqlTrans to the power dataset and displayed a sample of the transformed records.

In [49]:
#inspect of records of sqlTrans
transformed_power = sqlTrans.transform(power)
transformed_power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335

- We apply a binarization step to the Hour variable using a cutoff of 6.5, creating a new binary feature, night_vs_day, to indicate nighttime (Hour < 6.5) versus daytime (Hour ≥ 6.5) for downstream modeling.
    

In [50]:
from pyspark.ml.feature import Binarizer

binaryTrans = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="night_vs_day")

#inspect the transformer results
binary = binaryTrans.transform(
    sqlTrans.transform(power))
binary.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0

### One‑Hot encoding for the Month variable
Although the Month variable is stored as a LongType, it represents a categorical feature rather than a continuous numeric value. Therefore, we apply one-hot encoding to the `Month` column to generate binary indicator variables, allowing the model to capture seasonal effects without assuming an ordinal or linear relationship among months.

In [51]:
from pyspark.ml.feature import OneHotEncoder, VectorAssembler

# Location one-hot encoding
month_encoder = OneHotEncoder(inputCol="Month", outputCol="Month_ind", dropLast=True)

- We fit the month_encoder model to inspect the transformed records to ensure that the one-hot encoding is performed correctly.

In [52]:
#inspect the one-hot encode results
model = month_encoder.fit(binary)
encoded = model.transform(binary)
encoded.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|     Month_ind|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|       

### PAC on environmental variables 
- Principal Component Analysis (PCA) is used to reduce the dimensionality of the environmental variables Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows. The variables are assembled into a feature vector and standardized using [StandardScaler](https://www.sparkcodehub.com/pyspark/mllib/standard-scaler) to mitigate the influence of variables with larger magnitudes. PCA is then applied to extract two uncorrelated principal components, which provide lower‑dimensional representations for subsequent analysis and modeling.


In [53]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

#assemble raw features
assembler = VectorAssembler(inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol="pcaFeatures")

# Scale features (mean=0, std=1)
scaler = StandardScaler(inputCol="pcaFeatures", outputCol="scaledFeatures", withMean=True, withStd=True)

#PCA on scaled features
pca = PCA(k=2, inputCol="scaledFeatures", outputCol="pcaOutput")

#inspect the results
#set up pipeline
pipeline = Pipeline(stages=[assembler, scaler, pca])

# Fit and transform
pca_model = pipeline.fit(encoded)
pca_transformed = pca_model.transform(encoded)

pca_transformed.select("scaledFeatures", "pcaOutput").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------+---------------------------------------+
|scaledFeatures                                                                                      |pcaOutput                              |
+----------------------------------------------------------------------------------------------------+---------------------------------------+
|[-2.1079477438809433,0.3542085264292604,-0.7996341497278044,-0.6900839531060153,-0.6025312519793238]|[2.0690743213463727,0.8078678829472261]|
|[-2.1328903699941857,0.3991947174962608,-0.7996341497278044,-0.6900121009460847,-0.6028048802947571]|[2.1029210638806544,0.8152476222806391]|
|[-2.1502641992178924,0.3991947174962608,-0.800911098051248,-0.690042354487108,-0.6026841619203012]  |[2.1120662633791016,0.8227151924829956]|
|[-2.183291676554047,0.4313277111155467,-0.7996341497278044,-0.6899326854008982,-0.6027163534868227] |[2.14350318474222,0.8329135817940965]  |

As the output above, we combined several environmental measurements into a single dataset and used PCA to compress them into two summary variables that capture most of the information. These summaries were then used instead of the original variables in later analyses.

### Create label column
- we will rename our response variable `Power_Zone_3` as label

In [54]:
sqlLabel = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

- Use `VectorAssembler()` to put all predictors into a single features vector, which includes:   
    - two fitted PCA features (pcaOutput)      
    - a binary Hour variable (night_vs_day)   
    - Power_Zone_1   
    - Power_Zone_2   
    - Month indicator variables (Month_ind)   
Since the PCA features were standardized in previous steps, the additional non‑PCA features will also be standardized to ensure consistency across all predictors.

In [55]:
#assemble assembleFeatures
features_assembler = VectorAssembler(
    inputCols=["pcaOutput", "night_vs_day", "Power_Zone_1", "Power_Zone_2", "Month_ind"],
    outputCol="assembledFeatures"
)

#scale again after adding non‑PCA features.
final_scaler = StandardScaler(
    inputCol="assembledFeatures",
    outputCol="features",
    withMean=True,
    withStd=True
)

assembled_df = features_assembler.transform(pca_transformed)
scaler_model = final_scaler.fit(assembled_df)
featurestrans = scaler_model.transform(assembled_df)

# Inspect assembled features
featurestrans.select("features").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                                                                                                                                                                                              |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[1.355834830

From the output above, we confirmed that the transformation pipeline was completed and the the dataset is ready for modeling.

## Fit an Elastic Net Model
We next fit an Elastic Net regression model using the `CrossValidator()` and `LinearRegression()` function. Model hyperparameters are selected via cross-validation over a predefined grid consisting of all combinations of the following values:   

- regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

We are now ready to set up the pipeline, which includes transcormations and the model to be fitted. We use the `Pipeline()` function from the `pyspark.ml` module to set up a squential workflow of transformations/estimators.

In [56]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Elastic Net regression
lr = LinearRegression(featuresCol='features', labelCol='label') #create a liner regression instance

#define parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

- Set up the pipeline 

In [57]:
pipeline = Pipeline(stages=[
    sqlTrans,              # Cast hour 
    binaryTrans,           # night_vs_day
    month_encoder,         # Month one-hot encoding
    sqlLabel,              # Rename response variable
    assembler,             # Assemble environmental vars to create pcaFeatures
    scaler,                # Scale pcaFeatures
    pca,                   # PCA produces pcaOutput
    features_assembler,    # Combine pcaOutput + indicators to get assembledFeatures
    final_scaler,          # Scale final feature vector to features
    lr                     # Elastic Net regression model
])

We apply `CrossValidator()` to carry out 5-fold cross-validation over the specified hyperparameter grid. Model performance is evaluated using `RegressionEvaluator()`, with RMSE specified via the metricName argument. This procedure will split the dataset into five folds. In each iteration, the model is trained on four folds and validated on the remaining fold for every hyperparameter combination. The performance metric (rmse) is then averaged across all five folds, and the model with the lowest average rmse is selected as the best model.

In [58]:
from pyspark.ml.evaluation import RegressionEvaluator
#initialize cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    ),
    numFolds=5,        #5-fold cross-validation
    parallelism=32,
    seed = 42
)
#fitting the model, and choose the best set of parameters
cv_model = cv.fit(power)

26/04/27 23:53:11 ERROR LBFGS: Failure! Resetting history: breeze.optimize.FirstOrderException: Line search zoom failed


Evaluating the best model
- We will use the bestModel attribute to retrive the best model from cross validation
- Then, extract the final stage of the pipeline, the trained LinearRegression model, from the selected best model for further evaluation and interpretation.

In [59]:
# Retrieves the best pipeline model from the cross-validation
best_model = cv_model.bestModel
#Extracts the last stage of the pipeline
best_lr = best_model.stages[-1]  # LinearRegression is last stage

Report the optimal values of the tuning parameters (regParam and elasticNetParam) selected by cross‑validation using `getRegParam()` and `getElasticNetParam()` methods.

In [60]:
# Printing parameters for verification
print(f"Best regParam: {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

Best regParam: 0.05
Best elasticNetParam: 0.05


Both parameters were selected as 0.05 because cross‑validation identified a model with mild, Ridge‑dominant regularization that best balances bias and variance for the standardized feature set.

Report the CV error   
- The cross‑validation error is computed as the average RMSE across the five folds, evaluated over all hyperparameter combinations.

In [61]:
#report CV error
avg_rmse = min(cv_model.avgMetrics)
print(f"Best CV RMSE: {avg_rmse:.4f}")

Best CV RMSE: 2174.9962


Report training set RMSE
- The fitted regression model with the best parameters now has a transform method that can be used to make predictions using the entire dataset. The training set RMSE is obtained by applying the best fitted pipeline model to the entire dataset using the transform() method and evaluating prediction error using RMSE.

In [62]:
#apply the best fitted pipeline model to the entire dataset
train_predictions = best_model.transform(power)

#define evaluator
evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    )
# Evaluate RMSE
train_rmse = evaluator.evaluate(train_predictions)
print(f"Training RMSE: {train_rmse:.4f}")

Training RMSE: 2174.4490


The similarity between the cross‑validation RMSE (2174.9962) and the training RMSE (2174.4490) demonstrates that the model achieves stable generalization with minimal overfitting.

The model outputs are used to generate predictions, from which a residual column is computed as the difference between the observed label and the predicted value. The `.withColumn()` method is used to create this residual. A resulting data frame is then displayed containing the residuals, the label column, and the model predictions.

In [63]:
from pyspark.sql.functions import col

residuals_df = train_predictions.withColumn("residual", col("label") - col("prediction"))

residuals_df.select( "label", "prediction", "residual",).show(10, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20936.226841006075|-695.2629810060753|
|20131.08434|18701.58646809512 |1429.4978719048813|
|19668.43373|18237.18946977339 |1431.2442602266092|
|18899.27711|17615.892216250577|1283.3848937494222|
|18442.40964|17017.381842683895|1425.0277973161064|
|18130.12048|16540.614302205628|1589.5061777943738|
|17945.06024|16114.684233894428|1830.376006105571 |
|17459.27711|15742.038334202174|1717.238775797825 |
|17025.54217|15282.557973009167|1742.9841969908339|
|16794.21687|14934.899321422645|1859.3175485773554|
+-----------+------------------+------------------+
only showing top 10 rows


## Conclusion
Using the Power dataset, an elastic net linear regression model was fitted through a pipeline and tuned via 5‑fold cross‑validation, with RMSE serving as the evaluation metric. The optimal regularization parameters were selected based on cross‑validated performance. The best‑fitted pipeline model was then applied to the entire dataset to compute the training set RMSE, which was 2174.449. Prediction residuals were subsequently calculated as the difference between the observed and predicted values.
During this process, transformations were applied to the hour and month variables, and all features were standardized prior to PCA and model fitting to mitigate the influence of variables with larger magnitudes. The use of a pipeline enabled efficient tracking of feature transformations and streamlined the modeling workflow, while Spark facilitated scalable and efficient model fitting. In the next section, methods for handling streaming data will be explored.

# Streaming Part


## Reading a stream
We set up a Structured Streaming read operation that monitors a specified folder for incoming CSV files, which created by .py file).

In [72]:
#create spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

- The schema from the power DataFrame is extracted and reused to ensure consistent column types when setting up the streaming CSV source. The header option is set to True so that all incoming files are correctly read with column headers.

In [73]:
#set up schema
power_schema = power.schema

#set up the stream df
stream_df = spark.readStream.schema(power_schema).option("header", True).csv("csv_files")

## Transform/Aggregation Step
The streaming dataset is processed using two separate transformations derived from the same input stream, following these steps:

- The first transformation applies the fitted pipeline model to generate prediction values, residuals are computed as the difference between the observed values and the model predictions.

In [74]:
from pyspark.sql.functions import col

# Apply the fitted feature-engineering pipeline and the model to streaming data
predictions_df = (
    best_model
        .transform(stream_df)
        .withColumn("residual", col("label") - col("prediction"))     #Create a residual column
        .select("label", "prediction", "residual")
)

- The second transformation operates on the original stream and modifies the response variable `Power_Zone_3` to be named label.

In [75]:
#2nd treansformation
label_df = stream_df.withColumnRenamed("Power_Zone_3", "label")

- The two transformed streams are then joined on the common label column using `join()` method.

In [76]:
joined_stream = predictions_df.join(label_df, on="label")

## Write the stream

The `.writeStream` method is used to output the transformed streaming data to the console in append mode. The `.start()` method initiates the streaming query and begins monitoring the input directory for new data. The five CSV files are added to the empty `csv_files` folder one at a time to be processed incrementally.

In [77]:
#Write the transformed streaming data to the console and start the streaming query
query = (
    joined_stream
        .writeStream
        .format("console") 
        .outputMode("append")
        .start()
)

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17965.16129| 15964.51614601076|2000.6451439892408|       11.8|    83.0|     0.081|                0.037|        0.122| 29878.46809| 17842.68293|    3|  23|
|10386.50602| 10967.25797108474|-580.7519510847396|       7.68|    79.3|     4.921|                0.059|        0.085| 23187.69231| 18189.66942|   11|   1|
|26640.72362|24809.066551959317| 1831.657068040684|      13.74|    70.1|     0.074|                13.46|        13.24

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17581.93548|15920.172924041039| 1661.7625559589615|      19.14|   41.39|     0.086|                760.0|         76.6| 32157.95745| 17871.95122|    3|  13|
|9301.899898|15838.684392261106| -6536.784494261106|      18.92|    82.6|      4.92|                193.9|        120.0| 34961.41593| 23845.32225|    9|   9|
| 25653.9759| 24600.61149989192|  1053.364400108083|      15.22|    73.3|     0.073|                0.081|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|24572.53012| 24668.30288248272| -95.77276248272028|      15.33|    70.2|     0.075|                59.64|         73.7| 42136.70886| 25725.22796|    1|  17|
| 10328.6747| 9843.823615993126| 484.85108400687386|      11.11|   69.24|     4.917|                0.022|         0.13| 21575.38462| 17642.97521|   11|   4|
|15260.79027| 12629.20463601166| 2631.5856339883394|      18.07|    92.2|      0.07|                0.059|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|    16800.0|17167.291329519525| -367.2913295195249|      13.21|   47.33|     0.083|                465.3|        484.3| 33915.94937| 21424.92401|    1|  14|
|26088.36923|27298.118557400838|-1209.7493274008375|      21.22|    87.0|     0.065|                0.066|        0.122| 41477.08609| 25667.77547|    6|   0|
|15770.60241|15500.936601397814|  269.6658086021853|       8.06|    79.7|     0.084|                0.081|       

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14457.83133|13675.593935690866| 782.2373943091352|      4.451|    79.8|     0.086|                 0.07|        0.134|  22596.4557| 13787.23404|    1|   3|
|23611.78683|23250.309179287968| 361.4776507120332|      26.15|    74.2|     4.906|                356.6|        190.3|  35154.5394| 27826.82154|    8|   9|
|27203.34728|27544.442945608724|-341.0956656087219|      25.29|    85.5|     4.924|                0.069|        0.137

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|9767.710843| 9336.382225202038|  431.3286177979626|      19.18|    83.0|     0.069|                0.066|        0.126| 21495.38462| 16795.04132|   11|   4|
|10698.79518|10424.949043863011|  273.8461361369882|      18.97|    77.7|     0.071|                0.051|        0.104| 23249.23077|  17371.4876|   11|   1|
|18548.36364|19374.971390878665| -826.6077508786657|      15.84|    86.5|     0.078|                 86.5|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|11199.02736|11145.511915842926| 53.515444157073944|      20.76|   65.28|     4.917|                392.0|        42.76| 29921.75055| 24714.52282|   10|  15|
|18983.32288|18523.069552823883|  460.2533271761167|      21.87|   40.39|     4.906|                227.8|        42.64|  28211.8535| 21641.81626|    8|   8|
|22787.21003|25334.711284384728|-2547.5012543847297|      27.62|   34.94|     4.907|                761.0|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25480.16736|29165.739367343485|-3685.572007343486|      27.24|   69.03|     4.907|                676.1|         91.5| 38463.78738| 26787.34177|    7|  10|
|29696.80251|26969.684950707666|2727.1175592923355|       28.6|   31.81|     4.908|                0.084|        0.144| 37040.44395| 27074.12883|    8|   0|
| 18737.3494|11759.542366459822| 6977.807033540177|      19.53|    70.7|     4.917|                252.4|        246.

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|47598.32636|  38791.9421777281| 8806.384182271897|      27.93|   61.19|     4.905|                24.64|         19.9| 51208.50498| 34868.35443|    7|  20|
|17152.77108|15671.322498640813|1481.4485813591855|       15.2|    78.3|      0.08|                0.037|        0.107| 33273.84615| 27728.92562|   11|  22|
|24016.06695|26247.977442320655|-2231.910492320654|       23.8|   54.93|     4.918|                0.091|        0.09

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|15921.29032|17588.559757771687| -1667.269437771687|      26.12|   38.34|     4.921|                730.0|        45.91| 33126.12766| 21025.60976|    3|  14|
|8794.650456|6582.2690469424815|  2212.381409057518|       18.7|    85.6|     0.067|                0.077|        0.093| 23292.07877| 18616.18257|   10|   7|
|35409.53975| 33300.68114179727| 2108.8586082027323|       25.2|    79.8|     4.918|                 25.9|      

When We start the query, We read in streaming datasets using the method that created in `stream.py` file. We will submit `stream.py` in a python console.


In [78]:
# Stop the streaming query after processing all input files
query.stop()